In [ ]:
## Imports

import pandas as pd
import numpy as np
import datetime as dt


from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score
)

from sklearn.metrics import silhouette_score

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


## Unsupervised Model

In [ ]:
plot_df = features_df2.dropna(subset=["regime_name"]).copy()

# 1) Regime over time
plt.figure(figsize=(12, 4))
plt.plot(plot_df.index, plot_df["regime_3class"], linewidth=1)
plt.title("Market Regimes Over Time")
plt.xlabel("Date")
plt.ylabel("Regime (0 = Low, 1 = Mid, 2 = High)")
plt.yticks([0, 1, 2], ["Low_Vol", "Mid_Vol", "High_Vol"])
plt.show()


# 2) Scatter plot of regimes over time
plt.figure(figsize=(12, 4))
plt.scatter(
    plot_df.index,
    plot_df["regime_3class"],
    c=plot_df["regime_3class"],
    cmap="viridis",
    s=8
)
plt.title("Market Regimes Over Time")
plt.xlabel("Date")
plt.ylabel("Regime (0 = Low, 1 = Mid, 2 = High)")
plt.yticks([0, 1, 2], ["Low_Vol", "Mid_Vol", "High_Vol"])
plt.show()


# 3) VIX colored by regime
plt.figure(figsize=(12, 5))

# Lighter VIX line (background)
plt.plot(
    plot_df.index,
    plot_df["VIX"],
    color="gray",
    linewidth=1,
    alpha=0.5
)

# Stronger, clearer colors + better labels
colors = {
    0: "#2ca02c",   # Low Vol (green)
    1: "#ff7f0e",   # Mid Vol (orange)
    2: "#d62728"    # High Vol (red)
}

labels = {
    0: "Low Vol",
    1: "Mid Vol",
    2: "High Vol"
}

for regime, color in colors.items():
    subset = plot_df[plot_df["regime_3class"] == regime]
    plt.scatter(
        subset.index,
        subset["VIX"],
        color=color,
        s=25,
        label=labels[regime]
    )

plt.title("Market Regimes vs VIX")
plt.xlabel("Date")
plt.ylabel("VIX")

# Cleaner legend placement
plt.legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
vix_data = [
    plot_df.loc[plot_df["regime_name"] == "Low_Vol", "VIX"].dropna(),
    plot_df.loc[plot_df["regime_name"] == "Mid_Vol", "VIX"].dropna(),
    plot_df.loc[plot_df["regime_name"] == "High_Vol", "VIX"].dropna()
]

plt.figure(figsize=(8, 5))
plt.boxplot(vix_data, tick_labels=["Low_Vol", "Mid_Vol", "High_Vol"])
plt.title("VIX by Regime")
plt.ylabel("VIX")
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Load data
features_df2 = pd.read_csv("/work/Features_volatility.csv")
features_df2["date"] = pd.to_datetime(features_df2["date"])
features_df2 = features_df2.sort_values("date").set_index("date")

rf = pd.read_parquet("/work/reg_base2.parquet").sort_index()
df = pd.read_parquet("/work/data_prep.parquet").sort_index()


# Step 2: Align datasets by common dates
common_idx = features_df2.index.intersection(rf.index)
features_df2 = features_df2.loc[common_idx].copy()
rf = rf.loc[common_idx].copy()


df = df.loc[df.index.intersection(features_df2.index)].copy()


# Step 3: Define 
# feature set
kmeans_cols = [
    "global_vol",
    "vol_slope",
    "vov",
    "VIX",
    "yield_spread",
    "SPY_vol_12",
    "SPY_vol_4"
]

# Keep only clustering features and drop missing rows
X_kmeans = features_df2[kmeans_cols].dropna().copy()


# Step 4: Fixed chronological split
# Based on Laura's note
train_end = "2015-12-31"
val_end = "2020-12-31"

X_train_raw = X_kmeans.loc[X_kmeans.index <= train_end].copy()
X_val_raw = X_kmeans.loc[(X_kmeans.index > train_end) & (X_kmeans.index <= val_end)].copy()
X_test_raw = X_kmeans.loc[X_kmeans.index > val_end].copy()

print("KMeans train shape:", X_train_raw.shape)
print("KMeans validation shape:", X_val_raw.shape)
print("KMeans test shape:", X_test_raw.shape)


# Step 5: Scale using train only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)


# Step 6: Evaluate candidate K values
K = range(2, 8)
rows = []
inertia_vals = []

for k in K:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)

    train_labels_k = km.fit_predict(X_train)
    val_labels_k = km.predict(X_val)
    test_labels_k = km.predict(X_test)

    inertia_vals.append(km.inertia_)

    train_counts = pd.Series(train_labels_k).value_counts()
    val_counts = pd.Series(val_labels_k).value_counts()
    test_counts = pd.Series(test_labels_k).value_counts()

    sil_score = silhouette_score(X_train, train_labels_k)

    rows.append({
        "k": k,
        "inertia": round(km.inertia_, 2),
        "silhouette_score": round(sil_score, 4),
        "train_clusters_present": train_counts.shape[0],
        "val_clusters_present": val_counts.shape[0],
        "test_clusters_present": test_counts.shape[0],
        "train_min_cluster_size": int(train_counts.min()),
        "val_min_cluster_size": int(val_counts.min()) if len(val_counts) > 0 else np.nan,
        "test_min_cluster_size": int(test_counts.min()) if len(test_counts) > 0 else np.nan
    })

k_eval_df = pd.DataFrame(rows).sort_values(
    by=["silhouette_score", "val_clusters_present", "val_min_cluster_size"],
    ascending=[False, False, False]
)

print("\nK evaluation table:")
print(k_eval_df)


# Step 7: Elbow plot
plt.figure(figsize=(8, 5))
plt.plot(list(K), inertia_vals, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method on Training Set")
plt.xticks(list(K))
plt.show()



# Step 8: Choose final K
#final_k = 2
final_k = 3

kmeans = KMeans(n_clusters=final_k, random_state=42, n_init=10)
train_labels = kmeans.fit_predict(X_train)
val_labels = kmeans.predict(X_val)
test_labels = kmeans.predict(X_test)


# Step 9: Add regime labels back to full dataframe
features_df2["regime"] = np.nan

features_df2.loc[X_train_raw.index, "regime"] = train_labels
features_df2.loc[X_val_raw.index, "regime"] = val_labels
features_df2.loc[X_test_raw.index, "regime"] = test_labels

features_df2["regime"] = pd.to_numeric(features_df2["regime"], errors="coerce")

# Track split membership
features_df2["dataset_split"] = pd.NA
features_df2.loc[X_train_raw.index, "dataset_split"] = "train"
features_df2.loc[X_val_raw.index, "dataset_split"] = "val"
features_df2.loc[X_test_raw.index, "dataset_split"] = "test"


# Step 10: Name regimes economically
# Lowest VIX cluster = Low_Vol
# Middle VIX cluster = Mid_Vol
# Highest VIX cluster = High_Vol

train_regime_summary = (
    features_df2.loc[X_train_raw.index]
    .groupby("regime")[["VIX", "global_vol"]]
    .mean()
    .sort_values("VIX")
)

print("\nTrain regime summary:")
print(train_regime_summary)

train_regime_order = train_regime_summary.index.tolist()

regime_map = {
    train_regime_order[0]: "Low_Vol",
    train_regime_order[1]: "Mid_Vol",
    train_regime_order[2]: "High_Vol"
}

features_df2["regime_name"] = features_df2["regime"].map(regime_map)

# 3-class mapping for supervised work
features_df2["regime_3class"] = features_df2["regime_name"].map({
    "Low_Vol": 0,
    "Mid_Vol": 1,
    "High_Vol": 2
})


# Step 11: Print regime summaries by split
print("\nTrain regime means:")
print(features_df2.loc[X_train_raw.index].groupby("regime_name")[["SPY_return", "VIX", "global_vol"]].mean())

print("\nValidation regime means:")
print(features_df2.loc[X_val_raw.index].groupby("regime_name")[["SPY_return", "VIX", "global_vol"]].mean())

print("\nTest regime means:")
print(features_df2.loc[X_test_raw.index].groupby("regime_name")[["SPY_return", "VIX", "global_vol"]].mean())

print("\nTrain regime counts:")
print(features_df2.loc[X_train_raw.index, "regime_name"].value_counts())

print("\nValidation regime counts:")
print(features_df2.loc[X_val_raw.index, "regime_name"].value_counts())

print("\nTest regime counts:")
print(features_df2.loc[X_test_raw.index, "regime_name"].value_counts())


# Step 12: ETF behavior by regime
etf_cols = ["XLF", "XLK", "XLE", "XLY", "XLP", "XLV", "XLI", "XLB", "XLU", "IYR"]

# Use pct_change if df contains price levels
etf_returns = df[etf_cols].pct_change()

sdf = etf_returns.join(features_df2[["regime_name"]], how="inner").dropna()

etf_returns = df[etf_cols].pct_change()
sdf = etf_returns.join(features_df2[["regime_name", "dataset_split"]], how="inner").dropna()

print("\nETF average returns by regime - Train:")
print(
    sdf[sdf["dataset_split"] == "train"]
    .groupby("regime_name")[etf_cols]
    .mean()
)

print("\nETF average returns by regime - Validation:")
print(
    sdf[sdf["dataset_split"] == "val"]
    .groupby("regime_name")[etf_cols]
    .mean()
)

print("\nETF average returns by regime - Test:")
print(
    sdf[sdf["dataset_split"] == "test"]
    .groupby("regime_name")[etf_cols]
    .mean()
)

# Step 13: Save labeled regime file
features_df2.to_parquet(f"regime_labeled_k_combine.parquet")

print(f"\nSaved file: regime_labeled_k_combine.parquet")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=059a5e93-4459-411b-9a58-2a578fd7a892' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>